# Домашнее задание по теме «Метрики классификации»

Сегодня ты будешь работать с метриками классификации: ROC-кривой, ROC-AUC, Log Loss и многоклассовыми случаями.



## Подготовка

In [ ]:
# Подгрузим сбалансированные данные
!gdown 1t05bQzB4sXJFOG8yIT8l77n7kGKX2joW

Downloading...
From: https://drive.google.com/uc?id=1t05bQzB4sXJFOG8yIT8l77n7kGKX2joW
To: /content/task_a_binary_balanced (1).csv
100% 2.28k/2.28k [00:00<00:00, 9.14MB/s]


In [ ]:
# Подгрузим данные с дисбалансом
!gdown 1sAuQpiKTT6YQpZ7OT4P_JnYgjjtgiu5-

Downloading...
From: https://drive.google.com/uc?id=1sAuQpiKTT6YQpZ7OT4P_JnYgjjtgiu5-
To: /content/task_b_real_binary_imbalanced.csv
100% 1.03M/1.03M [00:00<00:00, 87.2MB/s]


In [ ]:
# Подгрузим многоклассовые данные
!gdown 11jztdBQpL4JTyUFmD6TkgO9S-ApF1jMP

Downloading...
From: https://drive.google.com/uc?id=11jztdBQpL4JTyUFmD6TkgO9S-ApF1jMP
To: /content/task_c_multiclass.csv
100% 175k/175k [00:00<00:00, 109MB/s]


In [ ]:
import pandas as pd

a = pd.read_csv('/content/task_a_binary_balanced (1).csv')
b = pd.read_csv('/content/task_b_real_binary_imbalanced.csv')
c = pd.read_csv('/content/task_c_multiclass.csv')
a.head()

,true_label,predicted_label,prob_class_0,prob_class_1
0,0,0,0.99,0.01
1,0,1,0.00,1.00
2,0,0,0.99,0.01
3,1,1,0.18,0.82
4,0,0,1.00,0.00


## Описание наборов данных

В рамках этого домашнего задания ты будешь работать с тремя наборами данных. Все они — предсказания моделей на трёх разных датасетах с одинаковой структурой.

* `a` — предсказания модели на cбалансированных данных. Столбцы:

  * `true_label`, `predicted_label`, `prob_class_0` — предсказанная вероятность принадлежности объекта классу 0;
  * `prob_class_1` — предсказанная вероятность принадлежности объекта классу 1.
* `b` — предсказания модели на несбалансированных данных. Столбцы:
  * `true_label`, `predicted_label`, `prob_class_0` — предсказанная вероятность принадлежности объекта классу 0;
  * `prob_class_1` — предсказанная вероятность принадлежности объекта классу 1.
* `c` — предсказания модели для многоклассового случая. Столбцы:
  * `true_label`, `predicted_label`, `prob_class_0` — предсказанная вероятность принадлежности объекта классу 0;
  * `prob_class_1` — предсказанная вероятность принадлежности объекта классу 1.
 - Дальше по аналогии для каждого класса в изначальном датасете.

В этом домашнем задании не важно, из каких датасетов и с помощью каких моделей получены `a`, `b` и `c`. Но если тебе интересно, ниже представлен код их генерации.

## Генерация данных

In [ ]:
# from sklearn.datasets import load_breast_cancer
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import make_pipeline
# from sklearn.metrics import classification_report

# from sklearn.model_selection import StratifiedKFold
# from sklearn.datasets import fetch_openml
# import os

# # Breast Cancer Dataset — Task A
# def generate_real_task_a():
#     data = load_breast_cancer(as_frame=True)
#     X = data.data
#     y = data.target

#     X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=0)
#     model = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=0))
#     model.fit(X_train, y_train)
#     probs = model.predict_proba(X_test)
#     preds = model.predict(X_test)

#     df = pd.DataFrame({
#         'true_label': y_test.values,
#         'predicted_label': preds,
#         'prob_class_0': probs[:, 0],
#         'prob_class_1': probs[:, 1],
#     })
#     return df

# # Credit Card Fraud Detection — Task B
# def generate_real_task_b():
#     df = fetch_openml(name="creditcard", version=1, as_frame=True).frame
#     df = df.sample(frac=1, random_state=42)  # Shuffle
#     X = df.drop(columns=["Class"])
#     y = df["Class"].astype(int)

#     X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
#     model = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=42))
#     model.fit(X_train, y_train)
#     probs = model.predict_proba(X_test)
#     preds = model.predict(X_test)

#     df_out = pd.DataFrame({
#         'true_label': y_test.values,
#         'predicted_label': preds,
#         'prob_class_0': probs[:, 0],
#         'prob_class_1': probs[:, 1],
#     })
#     return df_out

# # MNIST — Task C
# def generate_real_task_c():
#     X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
#     y = y.astype(int)
#     X = X / 255.0
#     _, X_test, _, y_test = train_test_split(X, y, stratify=y, test_size=0.05, random_state=1)

#     model = RandomForestClassifier(n_estimators=100, random_state=1)
#     model.fit(X[:10000], y[:10000])
#     probs = model.predict_proba(X_test)
#     preds = model.predict(X_test)

#     df = pd.DataFrame({
#         'true_label': y_test,
#         'predicted_label': preds,
#     })
#     for i in range(10):
#         df[f'prob_class_{i}'] = probs[:, i]

#     return df

# df_real_task_a = generate_real_task_a()
# df_real_task_b = generate_real_task_b()
# df_real_task_c = generate_real_task_c()



Используй на практике теоретический материал из лонгрида, лекций и семинаров, вручную реализовав несколько метрик классификации.

### Задача 1 [3 балла]

**Цель**: вручную реализовать некоторые метрики классификации. Это поможет:
* глубже погрузиться в их особенности;
* исследовать, как они ведут себя в разных режимах и типах дисбаланса классов.

**Задание**:

1. Реализуй вручную (без использования `sklearn.metrics`) следующие метрики для бинарной классификации:
  - `LogLoss` (`Binary Cross-Entropy`) [0,5 балла]

   Напиши функцию `log_loss_binary(y_true, y_prob)`, которая возвращает среднюю кросс-энтропию по входным массивам. Обрати внимание на необходимость защиты от `log(0)`. C этим может помочь [np.clip()](https://numpy.org/doc/2.1/reference/generated/numpy.clip.html).
  - `ROC-кривая` [0,5 балла]

   Реализуй функцию `roc_curve_manual(y_true, y_prob)`, которая возвращает массивы `FPR`, `TPR` и порогов. Используй жёсткие предсказания при различных порогах.
  - `ROC-AUC` [0,5 балла]

   Реализуй функцию `roc_auc_manual(fpr, tpr)`, которая считает площадь под ROC с помощью метода [np.trapezoid()](https://numpy.org/devdocs/reference/generated/numpy.trapezoid.html).

2. Реализуй метрику `F1-score` для многоклассового случая.
Напиши функцию `f1_score_multiclass(y_true, y_pred, average='macro')`, поддерживающую следующие режимы:

  - `macro` — среднее по классам [0,5 балла];
  - `micro` — глобальные TP/FP/FN [0,5 балла];
  - `weighted` — среднее с учётом поддержки класса [0,5 балла].

**Правила выполнения**
* Нельзя использовать `sklearn.metrics`, `precision_score` и `roc_auc_score`.
* Разрешено использовать `numpy` и `matplotlib`.

In [ ]:
import numpy as np

# -----------------------------------------
# Бинарная классификация: логарифмическая потеря (Log Loss)
# -----------------------------------------
def log_loss_binary(y_true, y_prob, eps=1e-15):
    # Переводим в массивы
    y_true = np.array(y_true)
    y_prob = ... # Напиши код здесь: np.clip(..., eps, 1 - eps) — защита от log(0)

    # Формула логарифмической потери
    loss = ... # Напиши код здесь
    return float(...)  # Напиши код здесь

# -----------------------------------------
# ROC-кривая: FPR, TPR, пороги
# -----------------------------------------
def roc_curve_manual(y_true, y_prob):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    thresholds = np.r_[np.inf, np.sort(np.unique(y_prob))[::-1]] # Сортируем пороги по убыванию

    tpr_list = []
    fpr_list = []

    P = ... # Напиши код здесь: количество y_true == 1
    N = ... # Напиши код здесь: количество y_true == 0

    # Перебираем набор порогов вероятности от 0 до 1
    for thresh in thresholds:
        # Бинаризуем предсказания: если вероятность >= порога — предсказываем 1, иначе 0
        y_pred = ... # Напиши код здесь

        # TP (True Positive): количество объектов, где модель предсказала класс 1 и это действительно класс 1
        TP = ... # Напиши код здесь

        # FP (False Positive): количество объектов, где модель ошибочно предсказала класс 1 (а на самом деле класс 0)
        FP = ... # Напиши код здесь

        # TPR (True Positive Rate):
        # доля верно предсказанных положительных объектов среди всех настоящих положительных
        # P — это общее количество объектов класса 1
        TPR = ... # Напиши код здесь

        # FPR (False Positive Rate):
        # доля ошибочно предсказанных положительных среди всех настоящих отрицательных
        # N — это общее количество объектов класса 0
        FPR = ... # Напиши код здесь

        # Сохраняем значения для построения ROC-кривой
        tpr_list.append(TPR)
        fpr_list.append(FPR)


    return np.array(fpr_list), np.array(tpr_list), thresholds

# -----------------------------------------
# ROC-AUC через метод трапеций
# -----------------------------------------
def roc_auc_manual(fpr, tpr):
    return ... # Напиши код здесь

# -----------------------------------------
# F1-метрика для многоклассовой классификации
# -----------------------------------------
def f1_score_multiclass(y_true, y_pred, average="macro"):
    # Преобразуем входные списки в NumPy-массивы для удобства вычислений
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Получаем все уникальные классы из объединения y_true и y_pred
    labels = np.unique(np.concatenate([y_true, y_pred]))

    # Списки для хранения F1-оценок по каждому классу и числа объектов этого класса
    f1_scores = []
    supports = []

    # Инициализация общих TP, FP и FN для микровзвешенной метрики
    TP_micro = 0
    FP_micro = 0
    FN_micro = 0

    # Перебираем каждый уникальный класс
    for label in labels:
        # TP: предсказали класс label, и он действительно такой
        TP = ... # Напиши код здесь

        # FP: предсказали класс label, но на самом деле класс другой
        FP = ... # Напиши код здесь

        # FN: на самом деле класс label, но модель предсказала другой
        FN = ... # Напиши код здесь

        # Общее количество примеров этого класса в выборке
        support = ... # Напиши код здесь

        # Precision (точность) для текущего класса
        precision = ... # Напиши код здесь

        # Recall (полнота) для текущего класса
        recall = ... # Напиши код здесь

        # F1-оценка для текущего класса
        f1 = ... # Напиши код здесь

        # Сохраняем результаты
        f1_scores.append(f1)
        supports.append(support)

        # Обновляем суммы TP, FP и FN для микроусреднения
        TP_micro += TP
        FP_micro += FP
        FN_micro += FN

    # Возвращаем финальную метрику в зависимости от выбранного режима усреднения
    if average == "macro":
        # Простое среднее по всем классам, без учёта их частоты
        return ... # Напиши код здесь

    elif average == "weighted":
        # Взвешенное среднее: учитываем, сколько примеров у каждого класса (support)
        return ... # Напиши код здесь

    elif average == "micro":
        # Общая точность и полнота по всем классам как единый бинарный случай
        precision = ... # Напиши код здесь
        recall = ... # Напиши код здесь
        return ... # Напиши код здесь

    else:
        # Ошибка, если передан неподдерживаемый тип усреднения
        raise ValueError("Неподдерживаемый режим усреднения")


Проверь в ячейке ниже, корректно ли отработали функции. Если да, то ячейка выведет 3 строки без ошибок, а разница между `sklearn` и самописной версией в 1-й и 2-й строках будет небольшой (в пределах 0,009).

In [ ]:
from sklearn.metrics import (
    log_loss as skl_log_loss,
    roc_curve as skl_roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    f1_score
)

# -----------------------------------------
# Бинарная классификация
# -----------------------------------------
y_true_bin = a['true_label']
y_prob_bin = a['prob_class_1']

# Log Loss
manual_log_loss = log_loss_binary(y_true_bin, y_prob_bin)
sklearn_log_loss = skl_log_loss(y_true_bin, y_prob_bin)
print('Разница LogLoss', manual_log_loss - sklearn_log_loss)

# ROC Curve & AUC
fpr_manual, tpr_manual, _ = roc_curve_manual(y_true_bin, y_prob_bin)
fpr_sklearn, tpr_sklearn, _ = skl_roc_curve(y_true_bin, y_prob_bin)
auc_manual = roc_auc_manual(fpr_manual, tpr_manual)
auc_sklearn = roc_auc_score(y_true_bin, y_prob_bin)
print('Разница AUC', auc_manual - auc_sklearn)

# -----------------------------------------
# Мультиклассовая классификация
# -----------------------------------------
y_true_multi = c['true_label']
y_pred_multi = c['predicted_label']

# F1 Macro
f1_macro_manual = f1_score_multiclass(y_true_multi, y_pred_multi, average="macro")
f1_macro_sklearn = f1_score(y_true_multi, y_pred_multi, average="macro")
assert np.isclose(f1_macro_manual, f1_macro_sklearn, atol=1e-6)

# F1 Micro
f1_micro_manual = f1_score_multiclass(y_true_multi, y_pred_multi, average="micro")
f1_micro_sklearn = f1_score(y_true_multi, y_pred_multi, average="micro")
assert np.isclose(f1_micro_manual, f1_micro_sklearn, atol=1e-6)

# F1 Weighted
f1_weighted_manual = f1_score_multiclass(y_true_multi, y_pred_multi, average="weighted")
f1_weighted_sklearn = f1_score(y_true_multi, y_pred_multi, average="weighted")
assert np.isclose(f1_weighted_manual, f1_weighted_sklearn, atol=1e-6)

print("Все ручные реализации многоклассовых F1 совпадают с результатами sklearn.")

### Задача 2 [3 балла]

**Цель**: оценить, как искажения в данных и предсказаниях влияют на поведение ROC-кривой и итоговый AUC.

**Заданиe**: воспользуйся данными `b` (сильный дисбаланс классов) и самописными реализациями `roc_curve_manual` и `roc_auc_manual` из задачи 1.

На одном графике должно быть 4 ROC-кривых:

1. Построй ROC-кривую и посчитай AUC, используя данные из `b` [0,5 балла].

2. Добавь шум [1 балл]:
 - Добавь гауссовский шум (например, `np.random.normal(0, 0.1, size)`) к вероятностям.
 * Обрежь значения в диапазоне `[0, 1]`.
 * Построй ROC и посчитай AUC после искажения.
 * Сравни с изначальными значениями.
3. Инвертируй истинные метки, сохранив исходные вероятности [0,5 балла]:
 * Инвертируй метки `y_test = 1 - y_test`.
 * Построй ROC и посчитай AUC после искажения.
 * Интерпретируй, что и почему изменилось.
4. Примени `Undersampling` отрицательного класса [1 балл]:
 * Семплируй из `y_test == 0` случайным образом меньшую часть (например, 20%).
 * Оставь все `y == 1`.
 * Построй ROC и посчитай AUC после искажения.
 * Интерпретируй, что и почему изменилось.

**Правила выполнения**
* В итоге должны получиться 4 ROC-кривые и их AUC-значения.
* В завершение напиши краткий анализ: как каждая трансформация влияет на конечный результат и насколько ROC чувствительна к шуму, дисбалансу классов и инверсии истинных меток.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

y_prob = b['prob_class_1']
y_test = b['true_label']

# ROC и AUC до искажений
fpr_orig, tpr_orig, _ = ... # Напиши код здесь
auc_orig = ... # Напиши код здесь

# Добавление шума
np.random.seed(42)
y_prob_noisy = np.clip(y_prob + np.random.normal(0, 0.1, size=y_prob.shape), 0, 1)
fpr_noise, tpr_noise, _ = ... # Напиши код здесь
auc_noise = ... # Напиши код здесь

# Инверсия меток при неизменных вероятностях
y_test_inv = 1 - y_test
fpr_inv, tpr_inv, _ = ... # Напиши код здесь
auc_inv = ... # Напиши код здесь

# Undersampling негативного класса
pos_idx = y_test == 1
neg_idx = y_test == 0

undersampled_neg = np.random.choice(np.where(neg_idx)[0], size=int(0.2 * neg_idx.sum()), replace=False)
selected_idx = ... # Напиши код здесь
y_test_us = ... # Напиши код здесь
y_prob_us = ... # Напиши код здесь

fpr_us, tpr_us, _ = ... # Напиши код здесь
auc_us = ... # Напиши код здесь

# Визуализация


... # Напиши код здесь


**Пример возможного анализа**

ROC-AUC не устойчив / устойчив к дисбалансу классов. После undersmapling AUС равен ..., убрано около 80% отрицательных примеров.
Вывод: ROC-AUC не зависит / зависит от пропорции классов, потому что ...

Является ли высокий ROC-AUC однозначным индикатором того, что модель хорошая? Это спорный вопрос. Что может быть не так:

1. AUC показывает качество ранжирования, но не всегда — обобщающей способности.

 Высокий AUC необязательно означает хорошую работу модели на новых данных. Он может показать аналогичный результат и у переобученной модели, которая запомнила обучающую выборку.

2. Сильный дисбаланс данных может скрывать переобучение.

 Сам по себе дисбаланс классов не обеспечивает высокий ROC-AUC. Постоянный прогноз даёт ROC-AUC = 0,5 независимо от соотношения классов. При сильном дисбалансе дополнительно оценивай precision, recall и PR-AUC.

3. Слишком высокий AUC — повод насторожиться. Возможные «тревожные сигналы»:
 * очень простой пайплайн решения;
 * мало данных (например, после undersampling);
 * при этом AUC почти идеальный.

 Стоит проверить:

  * не переобучилась ли модель на класс-меньшинство;
  * нет ли утечки данных;
  * корректны ли метки и нет ли в них шума.



### Задача 3 [2,5 балла]

**Цель**:
* найти новый подход и выяснить, как логарифмическая функция потерь (`LogLoss`) штрафует уверенные ошибки;
* узнать, какое влияние отдельные примеры оказывают на итоговое значение Loss по всей выборке.

**Задание** [2,5 балла]:

1. Используя самописную функцию из задачи 1, вычисли Log Loss для всей выборки.
2. Отдельно рассчитай Log Loss для уверенно ошибочных примеров: истинный класс равен 1, а вероятность класса 1 меньше 0.2; либо истинный класс равен 0, а вероятность класса 1 больше 0.8
3. Найди топ-20 ошибок с наибольшим вкладом в `LogLoss`.

> **Примечание**
* Log Loss — одна из главных вероятностных функций потерь для задачи классификации.
* Она сильно штрафует уверенные ошибки (например, предсказание 0,01 при классе 1).
* Это делает её особенно полезной в задачах, где важна не только точность, но и достоверность вероятности.




In [ ]:
y_true = b['true_label']
y_pred_proba = b['prob_class_1']

# Весь Log Loss
total_loss = ... # Напиши код здесь

# Реализация вручную
eps = 1e-15  # Чтобы избежать log(0)
y_pred_proba_clipped = ... # Напиши код здесь

# Уверенно ошибочные примеры:
# - y_true == 1 и y_pred_proba < 0.2
# - либо y_true == 0 и y_pred_proba > 0.8

# Определим «уверенно ошибочные» случаи
predicted_class = (y_pred_proba >= 0.5).astype(int)
is_confidently_wrong = ... # Напиши код здесь

# Log loss только по ним
confident_loss = ... # Напиши код здесь

print(f"LogLoss на всей выборке: {total_loss:.4f}")
print(f"LogLoss на уверенно ошибочных примерах: {confident_loss:.4f}")


In [ ]:
# Топ-20 примеров с максимальным Log Loss

individual_losses = ... # Напиши код здесь

top_20_indices = ... # Напиши код здесь

print("Топ-20 примеров по LogLoss:")
for idx in top_20_indices[::-1]:  # От самого большого к меньшему
    print(f"Индекс: {idx}, Истинный класс: {y_true[idx]}, Предсказание: {y_pred_proba[idx]:.4f}, LogLoss: {individual_losses[idx]:.4f}")


Обрати внимание, как отрабатывает формула `LogLoss`: чем ниже вероятность истинного класса, тем сильнее функция потерь штрафует модель за такое предсказание.

### Задача 4 [3,5 балла]

**Цель**: использовать многоклассовые данные, чтобы рассмотреть, как многоклассовые метрики будут показывать себя в зависимости от способа агрегации предсказаний и структуры классов.

**Задание**: используй данные `c`.

1. Для каждого класса построй бинарную `One-vs-Rest` ROC-кривую (текущий класс против всех остальных), используя реализацию из задачи 1 [0,5 балла].
2. Для каждой ROC-кривой рассчитай её AUC [0,5 балла].
3. Агрегируй полученные AUC, вручную реализовав два способа агрегации [1,5 балла]:
 * Macro AUC — простое среднее по классам [0,5 балла].
 * Micro AUC [0,5 балла]:
    * возьми все предсказания и истинные бинарные метки для каждого класса;
    * запиши их в один список;
    * рассмотри этот список как одну большую бинарную задачу, а не несколько разных.
 * Рассчитай эти же значения через `roc_auc_score` в `sklearn` и сравни самописную и библиотечную реализации [0,5 балла].
4. Рассчитай `f1_macro`, `f1_micro`, `f1_weighted` для данных `с`, используя код из задачи 1 [0,5 балла].
5. Письменно ответь на вопрос [0,5 балла]: какая из метрик в этом задании наиболее устойчива в задачах дисбаланса?

Почему это важно? `F1-score`, `ROC AUC` и другие метрики в многоклассовых задачах — не просто числа. Важно понимать, что именно они усредняют и как они реагируют на дисбаланс классов. Это поможет правильно интерпретировать качество модели и принимать более осознанные решения.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, f1_score, confusion_matrix

y_true = c['true_label'].values
y_proba = c.drop(columns=['true_label','predicted_label']).values
classes = np.unique(y_true)

# Чтобы увидеть более репрезентативные результаты, добавим шум в имеющиеся данные
y_noisy = y_proba + np.random.normal(0, 0.45, size=y_proba.shape)
y_noisy = np.clip(y_noisy, 1e-15, 1)
y_noisy /= y_noisy.sum(axis=1, keepdims=True)

In [ ]:
# One-vs-rest ROC и AUC
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
aucs = []

for cls in classes:
    # Бинарные метки: текущий класс vs остальные
    y_bin = ... # Напиши код здесь
    y_score = y_noisy[:, cls]

    fpr, tpr, _ = ... # Напиши код здесь
    roc_auc = ... # Напиши код здесь
    aucs.append(roc_auc)

    plt.plot(fpr, tpr, label=f"Класс {cls} (AUC={roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC по классам")
plt.legend()
plt.grid(True)
plt.show()

В случае, если часть ниже выполнена корректно, `assert` ничего не выведут, так что показаны будут только значения `Macro` и `Micro AUC`.

In [ ]:
# Macro — просто среднее по классам
macro_auc = ... # Напиши код здесь

# Micro: объединяем все бинарные задачи
y_true_bin = np.zeros_like(y_noisy)
for i in range(len(y_true)):
    y_true_bin[i, y_true[i]] = 1

fpr_micro, tpr_micro, _ = ... # Напиши код здесь
micro_auc = ... # Напиши код здесь

from sklearn.metrics import roc_auc_score
sklearn_macro = ... # Напиши код здесь
sklearn_micro = ... # Напиши код здесь

# Сравнение
assert np.isclose(macro_auc, sklearn_macro, atol=1e-2)
assert np.isclose(micro_auc, sklearn_micro, atol=1e-2)

print(f"Macro AUC: {macro_auc:.3f}, Micro AUC: {micro_auc:.3f}")


In [ ]:
y_true_multi = c['true_label']
y_pred_multi = c['predicted_label']

f1_macro = f1_score_multiclass(y_true_multi, y_pred_multi, average="macro")
f1_micro = f1_score_multiclass(y_true_multi, y_pred_multi, average="micro")
f1_weighted = f1_score_multiclass(y_true_multi, y_pred_multi, average="weighted")

print(f"Macro F1: {f1_macro:.5f}, Micro F1: {f1_micro:.5f}, Weighted F1: {f1_weighted:.5f}")

Macro F1: 0.95905, Micro F1: 0.95943, Weighted F1: 0.95939


**Вопрос.** Какая из метрик в этом задании наиболее устойчива в задачах дисбаланса?

**Пример возможного ответа.** Из всех метрик, которые оценивают многоклассовую задачу дисбаланса, самая устойчивая — это …, потому что …

По результатам домашнего задания можно сделать следующие выводы:

1. Глубокое понимание метрик начинается с ручной реализации.

 Реализуя `ROC-кривую`, `ROC AUC`, PR-кривую и `LogLoss` вручную, можно не просто разобрать формулы, но и понять логику вычислений и интерпретации. Это повышает способность критически оценивать метрики вне зависимости от задачи.

2. Метрики ведут себя по-разному в многоклассовых задачах.

 На реальных примерах тебе удалось закрепить подходы к усреднению (`macro`, `micro`, `weighted`) и понять, как структура классов и дисбаланс могут искажать восприятие качества модели.

3. Метрики чувствительны к типу ошибок и искажениям прогноза.

 Эксперименты с различными искажениями показали:
 * как предсказания модели влияют на поведение модели;
 * почему важно понимать, какие именно ошибки допускает модель.

4. Дисбаланс классов требует осознанного выбора метрик.

 Сравнивая поведение разных метрик при неравномерных классах, можно убедиться в том, что не существует универсального показателя. Важно выбирать метрику под конкретную задачу и учитывать её ограничения.


